In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Standardized Paths
INPUT_FILE = '../resultados/comparativo_perfil_cenarios_qgis.csv'
OUTPUT_DIR = '../resultados/'
OUTPUT_PLOT = 'behavioral_transition_matrix.png'
OUTPUT_SUMMARY = 'behavioral_migration_summary.csv'

def generate_transition_analysis():
    # 1. Load the results file
    if not os.path.exists(INPUT_FILE):
        print(f"Error: {INPUT_FILE} not found.")
        return

    # Reading with ';' separator as defined in your GAMA script
    df = pd.read_csv(INPUT_FILE, sep=',')
    
    # Define standardized order for profiles (English labels for plot, but matching CSV values)
    profile_order = ['AMBIENTALISTA', 'MODERADO', 'PERDULARIO']
    plot_labels = ['Env.', 'Mod.', 'Was.']
    
    scenarios = sorted(df['NM_CENARIO'].unique())
    num_scenarios = len(scenarios)
    
    # 2. Setup Plot Grid (6 rows x 3 columns for 18 scenarios)
    fig, axes = plt.subplots(6, 3, figsize=(18, 25))
    axes = axes.flatten()
    
    migration_stats = []

    print(f"--- Processing {num_scenarios} Scenarios ---")

    for i, scenario in enumerate(scenarios):
        # Filter data for specific scenario
        df_scen = df[df['NM_CENARIO'] == scenario]
        
        # Create Transition Matrix (Cross-tabulation)
        # From: TP_COMPORTAMENTO (Initial) -> To: TP_NOVO_COMPORTAMENTO (Final)
        matrix = pd.crosstab(
            df_scen['TP_COMPORTAMENTO'], 
            df_scen['TP_NOVO_COMPORTAMENTO'],
            dropna=False
        ).reindex(index=profile_order, columns=profile_order, fill_value=0)
        
        # Calculate Migration Rate (%)
        total_agents = matrix.sum().sum()
        stable_agents = sum([matrix.iloc[j, j] for j in range(len(profile_order))])
        migration_rate = ((total_agents - stable_agents) / total_agents) * 100 if total_agents > 0 else 0
        
        migration_stats.append({
            'Scenario': scenario,
            'Migration_Rate_Pct': round(migration_rate, 2),
            'Total_Population': total_agents
        })

        # 3. Plot Heatmap for the scenario
        sns.heatmap(
            matrix, annot=True, fmt='d', cmap='YlGnBu', 
            ax=axes[i], cbar=False,
            xticklabels=plot_labels, yticklabels=plot_labels
        )
        
        axes[i].set_title(f"Scenario: {scenario}\n(Migration: {migration_rate:.2f}%)", fontweight='bold')
        axes[i].set_xlabel("Final Profile (Simulated)")
        axes[i].set_ylabel("Initial Profile (Baseline)")

    # Adjust layout and save
    plt.tight_layout()
    path_plot = os.path.join(OUTPUT_DIR, OUTPUT_PLOT)
    plt.savefig(path_plot, dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Save Summary CSV
    df_summary = pd.DataFrame(migration_stats)
    path_summary = os.path.join(OUTPUT_DIR, OUTPUT_SUMMARY)
    df_summary.to_csv(path_summary, index=False, sep=';')
    
    print(f"Success! Matrix plot saved: {path_plot}")
    print(f"Summary data saved: {path_summary}")
    print("\n--- Top 3 Scenarios with Highest Behavioral Shift ---")
    print(df_summary.sort_values(by='Migration_Rate_Pct', ascending=False).head(3))

if __name__ == "__main__":
    generate_transition_analysis()

--- Processing 18 Scenarios ---
Success! Matrix plot saved: ../resultados/behavioral_transition_matrix.png
Summary data saved: ../resultados/behavioral_migration_summary.csv

--- Top 3 Scenarios with Highest Behavioral Shift ---
  Scenario  Migration_Rate_Pct  Total_Population
5       CV               10.54             15297
8    CVIII               10.54             15297
4      CIX               10.47             15297
